In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import Window, functions as F


In [2]:
S3_BUCKET = "datalake-teste2"
S3_BRONZE = f"s3a://{S3_BUCKET}/bronze"
S3_SILVER = f"s3a://{S3_BUCKET}/silver"
S3_GOLD = f"s3a://{S3_BUCKET}/gold"

SPARK_MASTER = "local[*]"
SPARK_APP_NAME = "medallion-pipeline"

print(f"Bronze:  {S3_BRONZE}")
print(f"Silver:  {S3_SILVER}")
print(f"Gold:    {S3_GOLD}")

Bronze:  s3a://datalake-teste2/bronze
Silver:  s3a://datalake-teste2/silver
Gold:    s3a://datalake-teste2/gold


In [3]:
spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

26/09/03 01:09:55 WARN Utils: Your hostname, hugo resolves to a loopback address: 127.0.1.1; using 10.159.141.203 instead (on interface wlp2s0)
26/09/03 01:09:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hugo/Desktop/Project/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hugo/.ivy2/cache
The jars for the packages stored in: /home/hugo/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-69e63094-a80c-492e-b948-403236db1b97;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.261 in central
:: resolution report :: resolve 354ms :: artifacts dl 14ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.261 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 by [com.amazonaws#aws-java-sdk-bundle;1.12.261] in [default]
	------------------------------------------------------------------

In [4]:
df = spark.read.parquet(f"{S3_SILVER}/purchase_diario")
print(f"\n📥 {df.count()} linhas lidas de purchase_diario")


26/09/03 01:10:01 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties



📥 22 linhas lidas de purchase_diario


In [5]:
# --- 5. Intervalo de validade ------------------------------------------
    # Uma versão vale do dia do seu evento até a véspera do próximo evento
    # daquela compra. A última fica aberta (dt_fim NULL).
    #
    # Os intervalos cobrem todos os dias sem buraco nem sobreposição: entre
    # 24/01 e 04/02 o purchase_id=55 não teve evento algum, mas a versão de
    # 23/01 continua vigente nesse período. É isso que faz o BETWEEN responder
    # "como estava em qualquer data" sem precisar de window function.
versoes = Window.partitionBy("purchase_id").orderBy("transaction_date")

df = (df
      .withColumn("dt_inicio_validade", F.col("transaction_date"))
      .withColumn(
            "dt_fim_validade",
            F.date_sub(F.lead("transaction_date").over(versoes), 1),
        )
        .withColumn("is_current", F.col("dt_fim_validade").isNull())
)

In [9]:
columns_to_drop = ["hash_evento","fontes_no_dia"]

df.drop(*columns_to_drop)\
        .orderBy("purchase_id", "dt_inicio_validade") \
        .show(50, truncate=False)

+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+----------------+------------------+---------------+----------+
|transaction_datetime|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|subsidiary   |transaction_date|dt_inicio_validade|dt_fim_validade|is_current|
+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+----------------+------------------+---------------+----------+
|2023-01-20 22:02:00 |55         |15947   |5           |2023-01-20|2023-01-20  |852852     |696969    |10           |50.00         |NULL         |2023-01-20      |2023-01-20        |2023-01-22     |false     |
|2023-01-23 00:05:00 |55         |15947   |5           |2023-01-20|2023-01-20  |852852     |696969    |10           |50.00         |nacional     |2023-01-23    

O que está sendo feito:

- Salvando a tabela no s3 na gold

In [ ]:
df.write.mode("overwrite").partitionBy("transaction_date").parquet(f"{S3_GOLD}/purchase_historico")